In [ ]:
import pandas as pd
import pickle
import scipy.stats as ss
import math
import numpy as np
import datetime
import xgboost as xgb
import os
import sklearn
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor 
from sklearn.metrics import mean_squared_error
from tqdm import tqdm

import warnings
warnings.simplefilter(action='ignore', category=Warning)
pd.options.mode.chained_assignment = None  # default='warn'

In [ ]:
with open("data/raw/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
# Convert the 'Date' column to datetime if it's not already
data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')

In [ ]:
data.head()

## Gradient Boosted Regression Trees
### Tuning parameters:
* shrinkage/learning weight 𝜈: learning_rate
* debth of the simple trees 𝐿: max_depth
* the number of trees 𝐵 added to the ensemble model: n_estimators

## All firms

In [ ]:
#Define huber loss function to minimize in the validation step
def huber_loss_error(y_true, y_pred, delta=1.35):
    res = []
    for i in zip(y_true, y_pred):
        if abs(i[0]-i[1])<=delta:
            res.append(0.5*((i[0]-i[1])**2))
        else:
            res.append(delta*((abs(i[0]-i[1]) )-0.5*(delta**2)))
    return np.sum(res)

In [ ]:
# 60 years of data
start_year_train = 1957
start_year_val = 1975
start_year_test = 1987
end_year_test = 2016

#Empty containers to save results from each window
predictions = []
y_test_list =[]
dates = []
dic_r2_all = {}

# Model’s complexity: dictionary to save the number of predictors randomly considered as potential split variables over time
dic_max_depth_all = {}

# Grid of prespecified values for each hyperparameter  
param_grid = {"learning_rate": np.logspace(-1, 0, len(np.arange(-1, 1, 0.2)))}
#convert grid to list containing all posible hyperparameter combinations to iterate over
grid = list(ParameterGrid(param_grid))
# Empty container to save the objective loss function (huber loss error) for each hyperparameter combination
huber_loss = np.full((len(grid),1),np.nan, dtype = np.float32)

In [ ]:
# Loop over the years
for year in tqdm(range(start_year_test, end_year_test+1), desc='Processing', unit='item'):
    # Define the start and end dates for training and test
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)

    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')

    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
   # Select the data for training and test
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_train)]
    val_data = data[(data['DATE'] >= start_date_val) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    # Extract features and target variables
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1) 
    y_train = train_data['RET']  
    X_val = val_data.drop(['permno', 'DATE', 'RET'], axis=1)  
    y_val = val_data['RET'] 
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1)
    y_test = test_data['RET']

    learning_rates = np.logspace(-1, 0, len(np.arange(-1, 1, 0.2)))
    for i in range(len(grid)): 
        xgb_reg = xgb.XGBRegressor(objective='reg:squarederror', 
                            learning_rate= grid[i]['learning_rate'],
                            n_estimators= 100,  # 树的数量  
                            max_depth= 1,  # 树的最大深度
                            subsample= 0.3,  # 随机采样训练样本的比例  
                            colsample_bytree= 0.8,   #随机采样特征的比例  
                            tree_method='hist' # 树的构建方法  
                               ) 
          
        # 训练模型  
        xgb_reg.fit(X_train, y_train)  
        y_val_pred = xgb_reg.predict(X_val)  
        #calculate huber loss for each hyperparameter combination
        #set delta to 1.35 following Huber (2004)
        huber_loss[i,0]= huber_loss_error(y_val,y_val_pred, delta=1.35)
        
    #The optimal combination of hyperparameters is the one that causes the lowest loss 
    optim_param = grid[np.argmin(huber_loss)]

    #Fit again using the train and validation set and the optimal value for each hyperparameter 
    xgb_reg_final = xgb.XGBRegressor(objective='reg:squarederror',
                            learning_rate= optim_param['learning_rate'],
                            n_estimators= 100,  
                            max_depth= 1, 
                            subsample = 0.3, 
                            colsample_bytree = 0.8,  
                            tree_method= 'hist'
                                    )  
    xgb_reg_final.fit(X_train, y_train)  
    preds = xgb_reg_final.predict(X_test)
    
    #Save predictions, dates and the true values of the dependent variable to list
    predictions.append(preds)
    dates.append(y_test.index)
    y_test_list.append(y_test)
    
    #Calculate OOS model performance the for current window
    r2 = 1-sum(pow(y_test-preds,2))/sum(pow(y_test,2))
    #Save OOS model performance and the respective month to dictionary
    dic_r2_all["r2." + str(y_test.index)] = r2

    #Concatenate to get results over the whole OOS test period 
    predictions_all= np.concatenate(predictions, axis=0)
    y_test_list_all= np.concatenate(y_test_list, axis=0) 
    dates_all= np.concatenate(dates, axis=0)
    
    #Calculate OOS model performance over the entire test period in line with Gu et al (2020)
    R2OOS_GBR = 1-sum(pow(y_test_list_all-predictions_all,2))/sum(pow(y_test_list_all,2))
    print("R2OOS gradient boosted regression tree: ", R2OOS_GBR)